In [1]:
import os
import re
from datetime import datetime

import pandas as pd

In [2]:
# -------------------------------
# Paths
# -------------------------------

RAW_PATH = "../data/raw"
CLEAN_PATH = "../data/cleaned"

os.makedirs(CLEAN_PATH, exist_ok=True)

In [3]:
orders_df = pd.read_csv(os.path.join(RAW_PATH, "orders.csv"))

products_df = pd.read_csv(os.path.join(RAW_PATH, "products.csv"))

customers_df = pd.read_csv(os.path.join(RAW_PATH, "customers.csv"))

order_items_df = pd.read_csv(os.path.join(RAW_PATH, "order_items.csv"))

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


In [4]:
print("="*60)
print("Orders")
print("="*60)

orders_df.info()

print("\n")

print("="*60)
print("Products")
print("="*60)

products_df.info()

print("\n")

print("="*60)
print("Customers")
print("="*60)

customers_df.info()

print("\n")

print("="*60)
print("Order Items")
print("="*60)

order_items_df.info()

Orders
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     1000 non-null   int64  
 1   customer_id  950 non-null    float64
 2   region_code  1000 non-null   str    
 3   status       1000 non-null   str    
 4   order_date   1000 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 39.2 KB


Products
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    600 non-null    int64  
 1   product_name  600 non-null    str    
 2   category      600 non-null    str    
 3   subcategory   600 non-null    str    
 4   cost_price    600 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 23.6 KB


Customers
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data colu

In [5]:
print("="*50)
print("Missing Values")
print("="*50)

print("\nOrders")
print(orders_df.isnull().sum())

print("\nProducts")
print(products_df.isnull().sum())

print("\nCustomers")
print(customers_df.isnull().sum())

print("\nOrder Items")
print(order_items_df.isnull().sum())

Missing Values

Orders
order_id        0
customer_id    50
region_code     0
status          0
order_date      0
dtype: int64

Products
product_id      0
product_name    0
category        0
subcategory     0
cost_price      0
dtype: int64

Customers
customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
dtype: int64

Order Items
item_id             0
order_id            0
product_id          0
quantity            0
unit_price          0
discount_percent    0
dtype: int64


In [6]:
issues_report = {

    "missing_customer_ids": 0,

    "invalid_dates": 0,

    "invalid_emails": 0,

    "negative_quantities": 0,

    "invalid_order_references": 0,

    "product_name_changes": 0

}

In [7]:
def clean_orders(df):

    df = df.copy()

    fixed_dates = []

    invalid_dates = 0

    for value in df["order_date"]:

        try:

            dt = datetime.strptime(
                str(value),
                "%Y-%m-%d %H:%M:%S"
            )

        except:

            try:

                dt = datetime.strptime(
                    str(value),
                    "%d-%m-%Y"
                )

            except:

                dt = pd.NaT
                invalid_dates += 1

        fixed_dates.append(dt)

    df["order_date"] = fixed_dates

    missing = df["customer_id"].isna().sum()

    issues_report["missing_customer_ids"] = missing

    issues_report["invalid_dates"] = invalid_dates

    df["customer_id"] = df["customer_id"].fillna(-1).astype(int)

    return df

In [8]:
orders_clean = clean_orders(orders_df)

orders_clean.head()

,order_id,customer_id,region_code,status,order_date
0,1,77,NORTH,RETURNED,2025-04-25 02:33:10
1,2,495,WEST,PLACED,2025-12-26 07:20:25
2,3,169,WEST,RETURNED,2024-12-05 15:19:27
3,4,55,EAST,PLACED,2025-01-12 06:26:09
4,5,195,EAST,CANCELLED,2025-12-31 17:08:05


In [9]:
print("Missing Customer IDs :", issues_report["missing_customer_ids"])

print("Invalid Dates :", issues_report["invalid_dates"])

Missing Customer IDs : 50
Invalid Dates : 0


In [10]:
# ---------------------------------------
# Function: Clean Products
# ---------------------------------------

def clean_products(df):

    df = df.copy()

    changes = 0

    cleaned_names = []

    for name in df["product_name"]:

        original = str(name)

        # Remove leading and trailing spaces
        cleaned = original.strip()

        # Remove multiple spaces
        cleaned = " ".join(cleaned.split())

        # Convert to Title Case
        cleaned = cleaned.title()

        if cleaned != original:
            changes += 1

        cleaned_names.append(cleaned)

    df["product_name"] = cleaned_names

    issues_report["product_name_changes"] = changes

    return df

In [11]:
products_clean = clean_products(products_df)

products_clean.head(10)

,product_id,product_name,category,subcategory,cost_price
0,1,Oneplus 12,Electronics,Mobile,49796.78
1,2,Samsung Galaxy S24,Electronics,Mobile,44374.24
2,3,Jeans,Clothing,Women,18613.47
3,4,Long Walk To Freedom,Books,Biography,36357.30
4,5,Steve Jobs,Books,Biography,41701.28
5,6,The Alchemist,Books,Fiction,34545.01
6,7,Jeans,Clothing,Women,47532.70
7,8,Water Bottle,Home,Kitchen,18430.01
8,9,Knife Set,Home,Kitchen,13866.48
9,10,Macbook Air,Electronics,Laptop,33172.16


In [12]:
print("=" * 50)
print("Product Cleaning Summary")
print("=" * 50)

print("Product Names Modified :", issues_report["product_name_changes"])

Product Cleaning Summary
Product Names Modified : 136


In [13]:
comparison = pd.DataFrame({
    "Original": products_df["product_name"].head(10),
    "Cleaned": products_clean["product_name"].head(10)
})

comparison

,Original,Cleaned
0,OnePlus 12,Oneplus 12
1,SAMSUNG GALAXY S24,Samsung Galaxy S24
2,Jeans,Jeans
3,Long Walk to Freedom,Long Walk To Freedom
4,Steve Jobs,Steve Jobs
5,The Alchemist,The Alchemist
6,Jeans,Jeans
7,Water Bottle,Water Bottle
8,Knife Set,Knife Set
9,MacBook Air,Macbook Air


In [14]:
# ---------------------------------------
# Function: Validate Email Addresses
# ---------------------------------------

import re

def validate_emails(df):

    df = df.copy()

    # Simple email validation pattern
    email_pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    invalid_customers = []

    for _, row in df.iterrows():

        email = str(row["email"]).strip()

        if not re.match(email_pattern, email):

            invalid_customers.append({
                "customer_id": row["customer_id"],
                "customer_name": row["customer_name"],
                "email": email
            })

    issues_report["invalid_emails"] = len(invalid_customers)

    return pd.DataFrame(invalid_customers)

In [15]:
invalid_emails_df = validate_emails(customers_df)

invalid_emails_df.head()

,customer_id,customer_name,email
0,26,Shannon Jones,joshuawashington
1,111,David Davis,keyemily@example
2,240,Amy Martinez,kevinmartin
3,336,Linda Mueller,asmithexample.com
4,378,Heather Brown,brobinsonexample.org


In [16]:
print("=" * 50)
print("Email Validation Summary")
print("=" * 50)

print("Invalid Emails Found :", issues_report["invalid_emails"])

Email Validation Summary
Invalid Emails Found : 10


In [17]:
invalid_emails_df.to_csv(
    os.path.join(CLEAN_PATH, "invalid_emails.csv"),
    index=False
)

print("Invalid email report saved.")

Invalid email report saved.


In [18]:
# ---------------------------------------
# Function: Check Referential Integrity
# ---------------------------------------

def check_referential_integrity(orders_df, order_items_df):

    # Create a set of valid order IDs
    valid_order_ids = set(orders_df["order_id"])

    # Find order_items with invalid order_id
    invalid_orders = order_items_df[
        ~order_items_df["order_id"].isin(valid_order_ids)
    ].copy()

    issues_report["invalid_order_references"] = len(invalid_orders)

    return invalid_orders

In [19]:
invalid_orders_df = check_referential_integrity(
    orders_clean,
    order_items_df
)

invalid_orders_df.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent


In [20]:
print("=" * 50)
print("Referential Integrity Report")
print("=" * 50)

print(
    "Invalid Order References :",
    issues_report["invalid_order_references"]
)

Referential Integrity Report
Invalid Order References : 0


In [21]:
# ---------------------------------------
# Count Negative Quantities
# ---------------------------------------

negative_quantity_count = (
    order_items_df["quantity"] < 0
).sum()

issues_report["negative_quantities"] = negative_quantity_count

print("Negative Quantities :", negative_quantity_count)

Negative Quantities : 90


In [22]:
# ---------------------------------------
# Save Cleaned Files
# ---------------------------------------

customers_df.to_csv(
    os.path.join(CLEAN_PATH, "customers_clean.csv"),
    index=False
)

products_clean.to_csv(
    os.path.join(CLEAN_PATH, "products_clean.csv"),
    index=False
)

orders_clean.to_csv(
    os.path.join(CLEAN_PATH, "orders_clean.csv"),
    index=False
)

order_items_df.to_csv(
    os.path.join(CLEAN_PATH, "order_items_clean.csv"),
    index=False
)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.


In [25]:
# ---------------------------------------
# Cleaning Report
# ---------------------------------------

report_path = os.path.join(
    "../reports",
    "cleaning_report.txt"
)

os.makedirs("../reports", exist_ok=True)

with open(report_path, "w") as file:

    file.write("=" * 50 + "\n")
    file.write("DATA CLEANING REPORT\n")
    file.write("=" * 50 + "\n\n")

    file.write(
        f"Missing Customer IDs      : {issues_report['missing_customer_ids']}\n"
    )

    file.write(
        f"Invalid Date Formats      : {issues_report['invalid_dates']}\n"
    )

    file.write(
        f"Product Names Modified    : {issues_report['product_name_changes']}\n"
    )

    file.write(
        f"Invalid Emails            : {issues_report['invalid_emails']}\n"
    )

    file.write(
        f"Negative Quantities       : {issues_report['negative_quantities']}\n"
    )

    file.write(
        f"Invalid Order References  : {issues_report['invalid_order_references']}\n"
    )

print("Cleaning report generated successfully.")

Cleaning report generated successfully.


In [26]:
with open(report_path, "r") as file:
    print(file.read())

DATA CLEANING REPORT

Missing Customer IDs      : 50
Invalid Date Formats      : 0
Product Names Modified    : 136
Invalid Emails            : 10
Negative Quantities       : 90
Invalid Order References  : 0

